# Abaqus — reading a 3-D finite-element part

An **Abaqus `.inp`** file is the input deck for a structural finite-element model. This notebook reads one with
the `abaqus` driver and shows what is in it.

Like the CGNS example, it is a **3-D solid** mesh (made of brick-shaped elements), so we look at it as points
from above rather than as a flat colour-map.

## 1. Read the file

In [ ]:
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from cleopatra.config import Config

from pyramids.netcdf.ugrid import MeshDataset

DATA = Path("../../../examples/data/real-data")
inp = DATA / "abaqus" / "1.inp"

t0 = time.perf_counter()
ds = MeshDataset.read_file(inp, driver="abaqus")
secs = time.perf_counter() - t0
Config.set_matplotlib_backend()

print(f"read in {secs:.1f} s")
print("nodes:", ds.n_node)
print("faces:", ds.n_face)

A 94 MB deck reads in a few seconds: **752 666 nodes** and **332 200 faces** — scale is not a problem.

It is a **3-D solid** part: the elements are bricks (8 corner nodes each), the coordinates are the model's own
engineering units (there is no map/CRS), and the height of each point is kept as a node field **`node_z`**.

## 2. The part from above

We plot the nodes from straight above (the x–y plane) and colour them by height `z`. There are 750 000 nodes,
so we draw every 9th one — enough to see the shape without an unreadable solid blob.

In [ ]:
x = np.asarray(ds.mesh.node_x)
y = np.asarray(ds.mesh.node_y)
z = np.asarray(ds.masked_values("node_z"))

step = 9  # draw every 9th node
fig, ax = plt.subplots(figsize=(6, 6))
sc = ax.scatter(x[::step], y[::step], c=z[::step], cmap="viridis", s=2, linewidths=0)
plt.colorbar(sc, label="height z")
ax.set_aspect("equal")
ax.set_xlabel("x"); ax.set_ylabel("y")
ax.set_title("Abaqus part from above (coloured by height)")

The dense cluster in the centre is the main body of the part (higher `z`, in yellow-green); the sparser
outer points are lower features of the same 3-D object. This is the whole part seen from directly overhead.

## 3. Zoom in on a small patch

The whole part is too dense to see individual elements. `crop(...)` cuts out a small window, and on that small
piece the surface *can* be drawn as a filled colour-map of `node_z`.

In [ ]:
cx, cy = ds.mesh.face_centroids
mx, my = float(np.median(cx)), float(np.median(cy))
patch = ds.crop(bbox=(mx - 2, my - 2, mx + 2, my + 2))
print("faces in this patch:", patch.n_face)

glyph = patch.plot("node_z", cmap="terrain", colorbar=True, title="A small patch — surface height")
glyph.ax.set_aspect("equal")

On a small enough piece the mesh is a clean surface again: the colour is the height `node_z` across the
patch. (A filled map only works locally here — over the whole 3-D solid the brick layers overlap when flattened,
which is why we used points-from-above for the full part.)

## Takeaway

- The `abaqus` driver reads a large `.inp` deck (**94 MB, 752 666 nodes**) into an ordinary pyramids
  `MeshDataset` in a few seconds.
- The part is a **3-D solid** (brick elements, engineering coordinates), so the whole thing is best viewed as
  **points from above**; a flat colour-map of the full 3-D mesh is not meaningful.
- `crop(...)` cuts out a small patch, and on that local piece an ordinary filled `plot("node_z", ...)` works —
  every pyramids operation behaves the same on the cropped mesh as on the full one.